# Machine 1 Manufacturing Statistical Analysis

## Connect to database and read SQL table

In [ ]:
import pandas as pd
import pyodbc

In [ ]:
conn_str = (
    "Driver={ODBC Driver 17 for SQL Server};"
    "Server=localhost\\SQLEXPRESS;"
    "Database=IRLManufacturingDB;" 
    "Trusted_Connection=yes;"
)
conn = pyodbc.connect(conn_str)
query = 'SELECT * FROM gold.machine1'

df = pd.read_sql(query, conn)

## Data Exploration

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df.info()

Observations:
1. There are no NULL values.
2. Production_shift is a category but stands as an object datatype

In [ ]:
df.head(5)

Observations:

3. **The data granularity is hourly (total machine time = 3600s)**
4. There are rows with no total production (good + bad pieces)
5. There are rows with the whole hour planned as downtime (machine_scheduled_downtime = 3600s)

Actions:

1. Delete duplicated columns (*production_datetime*)
2. Create dimension or change data-types as necessary
3. In the cases where there are not produced pieces, and where the whole hour is scheduled as downtime
is clear that we don't need this info since in the first case we can't use this info to analize production related metrics, and in the second case the whole time was planned as downtime, so there's not valuable information there.

In [ ]:
# 1.

# There are not duplicated columns.

In [ ]:
# 2.

df['production_shift'] = df['production_shift'].astype('category')
df['production_year'] = df['production_datetime'].dt.year
df['production_month'] = df['production_datetime'].dt.month
# trunk datetime data to year-month format
nzero_df['production_year_month'] = nzero_df['production_datetime'].dt.strftime('%Y-%m')
nzero_df['production_year_month'] = nzero_df['production_year_month'].astype('category')
ordered_date_month = ['2024-01', '2025-01', '2026-01', '2024-02', '2025-02', '2026-02', '2024-03', '2025-03', '2026-03', '2024-04', '2025-04', '2026-04', '2024-05', '2025-05', '2026-05', '2024-06', '2025-06', '2026-06', '2024-07', '2025-07', '2026-07', '2024-08', '2025-08', '2026-08', '2024-09', '2025-09', '2026-09', '2024-10', '2025-10', '2026-10', '2024-11', '2025-11', '2026-11', '2024-12', '2025-12']

df['total_production_pieces'] = df['production_good_pieces'] + df['production_bad_pieces']
df['total_no_scheduled_downtime'] = df['machine_running_time'] + df['machine_idle_time'] + df['machine_breakdown_time']

In [ ]:
# 3.
# There are no NULL values but there are 0 values that we need to be careful with.
# Particularly where (1) there are not produced pieces at all and where 
# (2) the whole hour is scheduled as downtime.

nzero_df = df[(df['total_production_pieces'] != 0) & (df['total_no_scheduled_downtime'] != 0)]

## Exploratory Data Analysis

### Production

#### Production Magnitude Analysis

In [ ]:
# # pivot a bad-pieces x trunked-date table in order to apply describe() method
# bad_pieces_ym = nzero_df.pivot(index='production_datetime', columns='production_year_month', values='production_bad_pieces')
# bad_pieces_ym[ordered_date_month].describe()

In [ ]:
# Compare graphically the differences of total defective pieces on a year-month basis
plt.figure(figsize=(12,4), dpi=200)
ax3 = sns.boxplot(nzero_df, x='production_month', y='production_bad_pieces', hue='production_year', palette='Set2', flierprops={"markeredgecolor": "white"})

ax3.set_facecolor("#343837")          # Inner color
ax3.figure.set_facecolor("#343837")    # Outer color
ax3.set_xlabel('MONTH', color='tomato')
ax3.set_ylabel('TOTAL DEFECTIVE PIECES', color='tomato')
ax3.set_title('DEFECTIVE PIECES PRODUCED BY MONTH', color='tomato')
ax3.tick_params(labelcolor='white')
#ax3.set_ylim(0,50) #ZOOM IN

In [ ]:
# pivot a quality percent x trunked-date table in order to apply describe() method
quality_percent_ym = nzero_df.pivot(index='production_datetime', columns='production_year_month', values='production_quality_percent')
quality_percent_ym[ordered_date_month].describe()

In [ ]:
# Compare graphically the differences of quality percent on a year-month basis
plt.figure(figsize=(12,4), dpi=200)
ax12 = sns.boxplot(nzero_df, x='production_month', y='production_quality_percent', hue='production_year', palette='Set2', flierprops={"markeredgecolor": "white"})

ax12.set_facecolor("#343837")          # Inner color
ax12.figure.set_facecolor("#343837")    # Outer color
ax12.set_xlabel('MONTH', color='tomato')
ax12.set_ylabel('QUALITY PERCENTAGE', color='tomato')
ax12.tick_params(labelcolor='white')
ax12.set_title('QUALITY PCT BY MONTH', color='tomato')
#ax12.set_ylim(80,100) #ZOOM IN

In [ ]:
# pivot a bad-pieces x shift table in order to apply describe() method
bad_pieces_y_shift = nzero_df.pivot(index='production_datetime', columns=['production_year', 'production_shift'], values='production_bad_pieces')
bad_pieces_y_shift.describe()

In [ ]:
# Compare graphically the differences of total defective pieces on a shift basis
plt.figure(figsize=(12,4), dpi=200)
ax1 = sns.boxplot(nzero_df, x='production_shift', y='production_bad_pieces', hue='production_year', palette='Set2', flierprops={"markeredgecolor": "white"})

ax1.set_facecolor("#343837")          # Inner color
ax1.figure.set_facecolor("#343837")    # Outer color
ax1.set_xlabel('SHIFT', color='tomato')
ax1.set_ylabel('TOTAL DEFECTIVE PIECES', color='tomato')
ax1.set_title('DEFECTIVE PIECES PRODUCED BY SHIFT', color='tomato')
ax1.tick_params(labelcolor='white')
#ax1.set_ylim(0,50) #ZOOM IN

In [ ]:
# pivot a quality percent x shift table in order to apply describe() method
quality_percent_y_shift = nzero_df.pivot(index='production_datetime', columns=['production_year', 'production_shift'], values='production_quality_percent')
quality_percent_y_shift.describe()

In [ ]:
# Compare graphically the differences of quality percent on a shift basis
plt.figure(figsize=(12,4), dpi=200)
ax13 = sns.boxplot(nzero_df, x='production_shift', y='production_quality_percent', hue='production_year', palette='Set2', flierprops={"markeredgecolor": "white"})

ax13.set_facecolor("#343837")          # Inner color
ax13.figure.set_facecolor("#343837")    # Outer color
ax13.set_xlabel('SHIFT', color='tomato')
ax13.set_ylabel('QUALITY PERCENTAGE', color='tomato')
ax13.set_title('QUALITY PERCENTAGE BY SHIFT', color='tomato')
ax13.tick_params(labelcolor='white')
#ax13.set_ylim(80,100) #ZOOM IN

#### Time Analysis

In [ ]:
# Extract the necessary data (from the dataframe) to see the evolution of the quality percent on a year-month basis
monthly_quality = nzero_df.groupby(pd.Grouper(key='production_datetime', freq='MS'))['production_quality_percent'].mean()

In [ ]:
# See graphically the evolution of quality percent on a year-month basis
plt.figure(figsize=(12,4), dpi=200)
ax19 = sns.lineplot(monthly_quality, x=monthly_quality.index, y=monthly_quality)
ax19.axhline(y=90, color='yellow', linestyle="--", linewidth=1.5, label="Lower Level")
ax19.axhline(y=98, color='yellow', linestyle="--", linewidth=1.5, label="Upper Level")

ax19.set_facecolor("#343837")          # Inner color
ax19.figure.set_facecolor("#343837")    # Outer color
ax19.set_xlabel('DATE', color='tomato')
ax19.set_ylabel('QUALITY PERCENTAGE', color='tomato')
ax19.set_title('QUALITY PERCENTAGE OVER TIME', color='tomato')
ax19.tick_params(labelcolor='white')

### Machine-time

#### Machine-time Magnitude Analysis

In [ ]:
# # pivot a machine-breakdown time x trunked-date table in order to apply describe() method
# breakdown_time_ym = nzero_df.pivot(index='production_datetime', columns='production_year_month', values='machine_breakdown_time')
# breakdown_time_ym[ordered_date_month].describe()

In [ ]:
# Compare graphically the differences of machine-breakdown time on a year-monh basis
plt.figure(figsize=(12,4), dpi=200)
ax15 = sns.boxplot(nzero_df, x='production_month', y='machine_breakdown_time', hue='production_year', palette='Set2', flierprops={"markeredgecolor": "white"})

ax15.set_facecolor("#343837")          # Inner color
ax15.figure.set_facecolor("#343837")    # Outer color
ax15.set_xlabel('MONTH', color='tomato')
ax15.set_ylabel('MACHINE BREAKDOWN TIME', color='tomato')
ax15.set_title('MACHINE BREAKDOWN TIME BY MONTH', color='tomato')
ax15.tick_params(labelcolor='white')
#ax15.set_ylim(0,200) #ZOOM IN

In [ ]:
# # pivot a machine-availability percent x trunked-date table in order to apply describe() method
# availability_time_ym = nzero_df.pivot(index='production_datetime', columns='production_year_month', values='machine_availability_percent')
# availability_time_ym[ordered_date_month].describe()

In [ ]:
# Compare graphically the differences of machine-availability percent on a year-monh basis
plt.figure(figsize=(12,4), dpi=200)
ax16 = sns.boxplot(nzero_df, x='production_month', y='machine_availability_percent', hue='production_year', palette='Set2', flierprops={"markeredgecolor": "white"})

ax16.set_facecolor("#343837")          # Inner color
ax16.figure.set_facecolor("#343837")    # Outer color
ax16.set_xlabel('MONTH', color='tomato')
ax16.set_ylabel('AVAILABILITY PERCENTAGE', color='tomato')
ax16.tick_params(labelcolor='white')
ax16.set_title('AVAILABILITY PCT BY MONTH', color='tomato')
#ax16.set_ylim(80, 100) # ZOOM IN

In [ ]:
# # pivot a machine-breakdown time x shift table in order to apply describe() method
# breakdown_percent_shift = nzero_df.pivot(index='production_datetime', columns=['production_year', 'production_shift'], values='machine_breakdown_time')
# breakdown_percent_shift.describe()

In [ ]:
# Compare graphically the differences of machine-breakdown time on a shift basis
plt.figure(figsize=(12,4), dpi=200)
ax17 = sns.boxplot(nzero_df, x='production_shift', y='machine_breakdown_time', hue='production_year', palette='Set2', flierprops={"markeredgecolor": "white"})

ax17.set_facecolor("#343837")          # Inner color
ax17.figure.set_facecolor("#343837")    # Outer color
ax17.set_xlabel('SHIFT', color='tomato')
ax17.set_ylabel('BREAKDOWN TIME', color='tomato')
ax17.set_title('BREAKDOWN TIME BY SHIFT', color='tomato')
ax17.tick_params(labelcolor='white')
#ax17.set_ylim(0, 200) # ZOOM IN

In [ ]:
# # pivot a machine availability percent x shift table in order to apply describe() method
# availability_percent_shift = nzero_df.pivot(index='production_datetime', columns=['production_year', 'production_shift'], values='machine_availability_percent')
# availability_percent_shift.describe()

In [ ]:
# Compare graphically the differences of machine-availability percent on a shift basis
plt.figure(figsize=(12,4), dpi=200)
ax18 = sns.boxplot(nzero_df, x='production_shift', y='machine_availability_percent', hue='production_year', palette='Set2', flierprops={"markeredgecolor": "white"})

ax18.set_facecolor("#343837")          # Inner color
ax18.figure.set_facecolor("#343837")    # Outer color
ax18.set_xlabel('SHIFT', color='tomato')
ax18.set_ylabel('AVAILABILITY PCT', color='tomato')
ax18.set_title('AVAILABILITY PCT BY SHIFT', color='tomato')
ax18.tick_params(labelcolor='white')
ax18.set_ylim(80, 100) # ZOOM IN

#### Time Analysis

In [ ]:
# Extract the necessary data (from the dataframe) to see the evolution of the availability percent on a shift basis
monthly_availability = nzero_df.groupby(pd.Grouper(key='production_datetime', freq='MS'))['machine_availability_percent'].mean()

In [ ]:
# See graphically the evolution of the availability percent on a shift basis
plt.figure(figsize=(12,4), dpi=200)
ax20 = sns.lineplot(monthly_availability, x=monthly_availability.index, y=monthly_availability)
ax20.axhline(y=60, color='yellow', linestyle="--", linewidth=1.5, label="Lower Level")
ax20.axhline(y=80, color='yellow', linestyle="--", linewidth=1.5, label="Upper Level")
ax20.axhline(y=90, color='green', linestyle="--", linewidth=1.5, label="World-Class Level")

ax20.set_facecolor("#343837")          # Inner color
ax20.figure.set_facecolor("#343837")    # Outer color
ax20.set_xlabel('DATE', color='tomato')
ax20.set_ylabel('AVAILABILITY PCT', color='tomato')
ax20.set_title('AVAILABILITY PCT OVER TIME', color='tomato')
ax20.tick_params(labelcolor='white')